# 03 - Leitura de Arquivos CSV

## Objetivo
Aprender a carregar dados de arquivos CSV e outros formatos.

## Conceitos

### pd.read_csv
Funcao principal para ler CSV. Parametros importantes:
- `sep`: delimitador (virgula por padrao).
- `header`: linha do cabecalho.
- `names`: nomes das colunas.
- `index_col`: coluna a ser usada como indice.
- `usecols`: subconjunto de colunas.
- `dtype`: tipos explicitos.
- `parse_dates`: colunas de data.
- `na_values`: valores a tratar como NaN.
- `nrows`: numero de linhas a ler.
- `encoding`: codificacao do arquivo.

### Outros formatos
- `pd.read_excel`, `pd.read_json`, `pd.read_parquet`.
- `pd.read_sql` para bancos de dados.
- `df.to_csv`, `df.to_excel`, `df.to_json`.

### Boas praticas
- Sempre inspecionar com `head`, `info`, `describe` apos carregar.
- Definir `dtype` evita inferencia errada.
- `parse_dates` facilita analises temporais.
- Tratar `na_values` explicitamente.

### Encoding
CSVs brasileiros frequentemente usam `latin-1` ou `utf-8`.
Se aparecerem caracteres estranhos, ajuste `encoding`.

## Criando um CSV de exemplo
Antes de praticar a leitura, geramos um arquivo `funcionarios.csv`
a partir de um DataFrame. Usamos `index=False` para nao gravar o indice
do Pandas como uma coluna extra.

In [ ]:
import pandas as pd
import numpy as np
import os

# Criando um CSV de exemplo
dados = {
    "id": [1, 2, 3, 4, 5],
    "nome": ["Ana", "Bruno", "Carla", "Diego", "Elisa"],
    "idade": [22, 25, 23, 28, 21],
    "salario": [3500.0, 4200.5, 3900.0, 5100.0, 3300.0],
    "cidade": ["SP", "RJ", "MG", "SP", "RJ"],
    "admissao": ["2020-01-15", "2019-03-22", "2021-07-10",
                 "2018-11-05", "2022-02-28"],
}
df_orig = pd.DataFrame(dados)
df_orig.to_csv("funcionarios.csv", index=False)
print("CSV criado com sucesso.\n")

## Leitura basica
`pd.read_csv` infere o cabecalho e os tipos automaticamente.
Repare que a coluna `admissao` e lida como texto (`object`) por padrao.

In [ ]:
# Leitura basica
df = pd.read_csv("funcionarios.csv")
print("Leitura basica:\n", df)
print("Dtypes:\n", df.dtypes)

## parse_dates
Com `parse_dates=["admissao"]`, o Pandas converte a coluna para `datetime64`,
permitindo operacoes temporais (extract, resample, etc.).

In [ ]:
# Definindo tipo da coluna data
df = pd.read_csv("funcionarios.csv", parse_dates=["admissao"])
print("\nCom parse_dates:\n", df.dtypes)

## index_col
`index_col="id"` usa a coluna `id` como indice do DataFrame, em vez do
indice inteiro padrao (0, 1, 2, ...).

In [ ]:
# Definindo indice
df_idx = pd.read_csv("funcionarios.csv", index_col="id")
print("\nCom index_col='id':\n", df_idx)

## usecols
`usecols` carrega apenas as colunas desejadas, economizando memoria
e tempo de leitura em arquivos grandes.

In [ ]:
# Selecionando colunas
df_cols = pd.read_csv("funcionarios.csv",
                      usecols=["nome", "salario"])
print("\nuse_cols:\n", df_cols)

## nrows
`nrows` le apenas as primeiras N linhas — util para inspecionar
rapidamente arquivos muito grandes.

In [ ]:
# Limitar linhas
df_n = pd.read_csv("funcionarios.csv", nrows=3)
print("\nnrows=3:\n", df_n)

## dtype explicito
Podemos forcar os tipos de colunas especificas com `dtype={...}`.
Isso evita inferencia errada e reduz o consumo de memoria
(ex.: `int32` em vez de `int64`).

In [ ]:
# Definindo dtypes
df_dt = pd.read_csv("funcionarios.csv",
                    dtype={"idade": "int32", "salario": "float32"})
print("\ndtypes explicitos:\n", df_dt.dtypes)

## na_values
`na_values` define quais strings devem ser interpretadas como `NaN`.
Aqui incluimos `""`, `"NA"` e `"N/A"`. Como o CSV original nao tem
ausentes, a contagem de NaN deve ser zero.

In [ ]:
# Tratando valores ausentes
df_na = pd.read_csv("funcionarios.csv", na_values=["", "NA", "N/A"])
print("\nCom na_values:\n", df_na.isna().sum())

## Inspecao apos a leitura
Boas praticas: sempre conferir `info()`, `describe()` e `head()`
logo apos carregar um arquivo, para detectar problemas de tipos ou valores.

In [ ]:
# Inspecao apos leitura
print("\ninfo:")
df.info()
print("\ndescribe:\n", df.describe())
print("\nhead:\n", df.head())

## Salvando subconjuntos
Podemos filtrar o DataFrame (ex.: apenas funcionarios de SP) e salvar
o resultado em um novo CSV com `to_csv`.

In [ ]:
# Salvando subconjuntos
df[df["cidade"] == "SP"].to_csv("funcionarios_sp.csv", index=False)
print("\nArquivo funcionarios_sp.csv criado.")

# Lendo de volta
df_sp = pd.read_csv("funcionarios_sp.csv")
print("\nFiltrado SP:\n", df_sp)

## Exportando e lendo JSON
`to_json` com `orient="records"` produz uma lista de dicionarios (uma
entrada por linha). `read_json` reconstroi o DataFrame a partir do arquivo.

In [ ]:
# Exportando para JSON
df.to_json("funcionarios.json", orient="records", indent=2)
print("\nJSON criado.")

# Lendo JSON
df_json = pd.read_json("funcionarios.json")
print("\nLido do JSON:\n", df_json)

## Limpeza dos arquivos temporarios
Removemos os arquivos gerados para deixar o ambiente limpo.

In [ ]:
# Limpando arquivos gerados
for f in ["funcionarios.csv", "funcionarios_sp.csv", "funcionarios.json"]:
    if os.path.exists(f):
        os.remove(f)
print("\nArquivos temporarios removidos.")